# Week 2 Day 2

## Covered Today
1. Building Data Science UIs with Gradio (No Front-End Skills Required)
2. Building Your First Gradio Interface with Callbacks and Sharing
3. Building Gradio Interfaces with Authentication and GPT Integration
4. Markdown Responses and Streaming with Gradio and OpenAI
5. Building Multi-Model Gradio UIs with GPT and Claude Streaming

In [1]:
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display

### Now we load all the API Keys

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

### Now check if all loaded keys exist and are as per the format

In [3]:
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")


if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")


if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")


if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [4]:
# create clients for each provider
# for openai, we simply use OpenAI, for others we need to specify the base url and key, while using openai api library
openai_client = OpenAI()

google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'

In [5]:
google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

Now, we start with Gradio. Gradio is a library that enables creation of frontend UIs using python code. For this, we start by importing Gradio. And the convention is to > import gradio as gr. We start by installing the lib in our venv by using uv add gradio. 

Gradio added, now we import it.

In [6]:
import gradio as gr

/Users/home/projects/llm_engineering_practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# next we start by writing a simple call gpt function in which we can enter a prompt
system_message = "You are a helpful assistant. Respond in simple text(non markdown)"

In [8]:
def call_gpt(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': prompt}
    ]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages)
    return response.choices[0].message.content

In [9]:
call_gpt('How are you?')

"I'm doing well, thank you! How can I assist you today?"

Now we move to user interface

In [10]:
# we start with a simple function to understand how gradio works
def shout(text):
    print(f"Shout has been called with {text}")
    return text.upper()

In [11]:
shout("Hello")

Shout has been called with Hello


'HELLO'

In [12]:
gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never').launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
# next we try sharing gradio using the same last function
def shout(text):
    return text.upper()

In [14]:
gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never').launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://94d2ac83d0a4c4efd8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
# next we move to adding some more elements in the gradio interface rather than using the def version
message_input = gr.Textbox(label='Enter Message:', info="Enter a message to make it shouted", lines=7)
message_output = gr.Textbox(label='Response', lines=8)

view = gr.Interface(
    fn=shout,
    title='Shout',
    inputs=[message_input],
    outputs=[message_output],
    examples=['hello', 'howdy'],
    flagging_mode='never'
)
view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [16]:
# next up, we take the same interface details and implement for GPT conversation
message_input = gr.Textbox(label='Enter Prompt', lines=8)
message_output = gr.Textbox(label='Response', lines=8)

view = gr.Interface(
    inputs=[message_input],
    outputs=[message_output],
    title='Gpt Bot',
    fn=call_gpt,
    flagging_mode='never',
    examples=['How are you?', 'Tell me about transformers in LLMs']
)

view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [22]:
# next up, we make a tiny change, to show the output in markdown, for this, we make changes to system_message and rewrite the function (though we can do without it also, but just for practice)

system_message = 'You are a helpful assistant. You respond in markdown'

def call_gpt(prompt):
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': prompt}
    ]
    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages)

    return (response.choices[0].message.content)

In [18]:
# now we run the above once to check if all's working.
call_gpt('in one line tell me, what is a transformer in LLMs')

A transformer in LLMs is a neural network architecture that uses self-attention mechanisms to efficiently process and generate sequential data.

In [19]:
# now, since the above was asked in one line, hence, no markdown output came, now, lets run it again to check, markdown is working.
call_gpt("Tell me what are transformers in LLMs")

**Transformers** are a type of deep learning architecture that has become the foundation for many large language models (LLMs), such as GPT (Generative Pre-trained Transformer) series, BERT, and others. They have revolutionized natural language processing (NLP) tasks due to their efficiency and effectiveness in handling sequential data like text.

### What are Transformers?

Transformers are neural network architectures introduced in the paper **"Attention Is All You Need"** by Vaswani et al. in 2017. Unlike previous sequence models (such as RNNs or LSTMs), transformers rely entirely on a mechanism called **self-attention** to process input data.

### Key Components of Transformers

1. **Self-Attention Mechanism**  
   - Allows the model to weigh the importance of different words in the input sequence relative to each other.  
   - Helps capture context and relationships regardless of the distance between words in the sequence.
   
2. **Multi-Head Attention**  
   - Multiple attention mechanisms running in parallel, allowing the model to focus on different parts of the sequence from multiple perspectives.

3. **Positional Encoding**  
   - Since transformers don’t process data sequentially, positional encodings are added to input embeddings to give the model information about the order of tokens.
   
4. **Feedforward Neural Networks**  
   - Applied to each position separately and identically after the attention layers.

5. **Layer Normalization and Residual Connections**  
   - Techniques to facilitate training deeper networks and help with gradient flow.

### Why Transformers are Important for LLMs?

- **Parallelization:** Unlike RNNs, transformers do not require sequential processing of tokens, enabling training on much larger datasets efficiently.
- **Long-range Dependencies:** Self-attention allows transformers to capture context over long sequences better.
- **Scalability:** The architecture scales well with increasing data and model size, which is crucial for building large language models.

### In Summary

Transformers form the backbone of most modern large language models. Their ability to handle context, scale efficiently, and capture complex language patterns is why they have become the standard architecture for state-of-the-art NLP systems.

---

If you'd like, I can also provide a simple diagram or code example illustrating transformers!

In [23]:
# awesome, markdown works fine, now we just take this function and implement gradio UI on top of this
input_message = gr.Textbox(label='Ask your question', lines=8)
output_message = gr.Markdown(label='Answer')

view = gr.Interface(
    fn=call_gpt,
    inputs=[input_message],
    outputs=[output_message],
    title='Your personal AI ChatBot',
    flagging_mode='never',
    examples=['What are transformers in LLMs?', 'What is life?']
)

view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


As we see above, results are being published in markdown on the right, with proper formatting